In [0]:
from pyspark.sql import functions as F

orders_silver = spark.table(
    "workspace.fashion_silver.orders"
)

order_items_silver = spark.table(
    "workspace.fashion_silver.order_items"
)

products_silver = spark.table(
    "workspace.fashion_silver.products"
)

print("Product performance source tables loaded successfully.")

Product performance source tables loaded successfully.


In [0]:
delivered_orders = (
    orders_silver
    .filter(F.col("order_status") == "Delivered")
    .select("order_id", "order_date")
)

delivered_order_items = (
    order_items_silver
    .join(delivered_orders, on="order_id", how="inner")
    .join(
        products_silver.select(
            "product_id",
            "category",
            "subcategory",
            "brand",
            "product_status"
        ),
        on="product_id",
        how="inner"
    )
)

print("Delivered order items:", delivered_order_items.count())

Delivered order items: 417023


In [0]:
category_performance = (
    delivered_order_items
    .groupBy("product_category")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("gross_amount"), 2).alias("gross_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discounts"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue")
    )
    .orderBy(F.desc("net_revenue"))
)

category_performance.show(truncate=False)

+----------------+----------+------------+--------------+---------------+--------------+
|product_category|units_sold|total_orders|gross_revenue |total_discounts|net_revenue   |
+----------------+----------+------------+--------------+---------------+--------------+
|Ethnic Wear     |119700    |51973       |6.4909886374E8|3.989102529E7  |6.0920783845E8|
|Footwear        |120327    |52112       |4.5860867418E8|2.805063274E7  |4.3055804144E8|
|Women's Clothing|119985    |52029       |3.8517521103E8|2.369121055E7  |3.6148400048E8|
|Sportswear      |119413    |51928       |3.276859268E8 |1.995067965E7  |3.0773524715E8|
|Accessories     |120399    |52014       |3.1350469326E8|1.946714218E7  |2.9403755108E8|
|Men's Clothing  |113627    |49676       |3.0672271913E8|1.879427446E7  |2.8792844467E8|
|Kids' Clothing  |119804    |51903       |1.6878096983E8|1.031324923E7  |1.584677206E8 |
+----------------+----------+------------+--------------+---------------+--------------+



In [0]:
brand_performance = (
    delivered_order_items
    .groupBy("product_brand")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue")
    )
    .orderBy(F.desc("net_revenue"))
)

brand_performance.show(truncate=False)

+---------------+----------+------------+--------------+
|product_brand  |units_sold|total_orders|net_revenue   |
+---------------+----------+------------+--------------+
|Northstar      |85627     |38675       |2.6152744318E8|
|Thread Theory  |88376     |39870       |2.5842104591E8|
|Aura Studio    |84857     |38373       |2.4685581931E8|
|Vivid Vogue    |83597     |37825       |2.4572593582E8|
|House of Nova  |84214     |38070       |2.4450707247E8|
|The Cotton Edit|82254     |37295       |2.4360925683E8|
|Everyday Co.   |82249     |37214       |2.417864478E8 |
|Mode Avenue    |81239     |36854       |2.4148118687E8|
|Urban Loom     |81595     |37022       |2.3346381734E8|
|Nexa Wear      |79247     |35953       |2.3204081834E8|
+---------------+----------+------------+--------------+



In [0]:
product_performance = (
    delivered_order_items
    .groupBy(
        "product_id",
        "product_category",
        "product_subcategory",
        "product_brand"
    )
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("gross_amount"), 2).alias("gross_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discounts"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue")
    )
    .withColumn(
        "average_selling_price",
        F.round(F.col("net_revenue") / F.col("units_sold"), 2)
    )
    .orderBy(F.desc("net_revenue"))
)

product_performance.show(10, truncate=False)

+----------+----------------+-------------------+-------------+----------+------------+-------------+---------------+-----------+---------------------+
|product_id|product_category|product_subcategory|product_brand|units_sold|total_orders|gross_revenue|total_discounts|net_revenue|average_selling_price|
+----------+----------------+-------------------+-------------+----------+------------+-------------+---------------+-----------+---------------------+
|3748      |Ethnic Wear     |Ethnic Kurtas      |Aura Studio  |131       |64          |1245853.23   |58964.08       |1186889.15 |9060.22              |
|6816      |Ethnic Wear     |Lehengas           |Urban Loom   |122       |57          |1210785.34   |79395.76       |1131389.58 |9273.69              |
|6205      |Ethnic Wear     |Sarees             |Vivid Vogue  |133       |64          |1185873.22   |84259.39       |1101613.83 |8282.81              |
|4727      |Ethnic Wear     |Ethnic Kurtas      |Thread Theory|118       |59          |1

In [0]:
category_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_gold.category_performance")

brand_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_gold.brand_performance")

product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_gold.product_performance")

print("All three Gold product performance tables saved successfully.")

All three Gold product performance tables saved successfully.


In [0]:
for table_name in [
    "category_performance",
    "brand_performance",
    "product_performance"
]:
    df = spark.table(f"workspace.fashion_gold.{table_name}")
    print(f"{table_name}: {df.count()} records")

category_performance: 7 records
brand_performance: 10 records
product_performance: 10000 records
